# 01 - EDA das Cestas de Compra (Fase 2 · ML · Task 1)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho (Item2Vec)**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`

---

### Objetivo
Conhecer os dados que alimentam o modelo, verificar se são adequados e responder: **há dado suficiente, e com sinal suficiente, para treinar o Item2Vec?**

O Item2Vec trata cada pedido como uma "frase" e cada SKU como uma "palavra". Ele aprende que dois produtos são parecidos quando aparecem **nas mesmas cestas**. Por isso a EDA olha principalmente para as **cestas** e para a **co-ocorrência entre produtos**, e não só para as colunas isoladas.

### Fonte: somente o stream (tempo real), já na Silver
| Tabela | Caminho | Origem |
|---|---|---|
| Itens de pedido | `squad2/grupo2/silver/ecommerce_itens_pedido` | pipeline José & Tiago (Fase 1); já traz `dt_pedido` |
| Produtos | `squad2/silver/ecommerce_produtos` | pipeline Dupla 1 (Fase 1) |
| Categorias | `squad2/silver/ecommerce_categorias` | pipeline Dupla 1 (Fase 1) |

O histórico batch (`raw/batch-data`) **não é usado**: o modelo aprende e é avaliado com o mesmo catálogo e o mesmo comportamento de compra que vai atender no streaming.

### Perguntas
1. Volume, período e qualidade
2. Distribuições e outliers das colunas numéricas
3. Tamanho das cestas
4. Correlação entre as variáveis da cesta
5. Popularidade dos SKUs (cauda longa) e tamanho do vocabulário
6. Co-ocorrência: os produtos são comprados juntos de forma não aleatória?
7. Viabilidade do hold-out (teste do modelo)

In [ ]:
%pip install python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, credenciais e caminhos

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, lit

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

RAIZ_SQUAD = f"abfss://squad2@{fqdn}"
FONTES = {
    "itens":      f"{RAIZ_SQUAD}/grupo2/silver/ecommerce_itens_pedido",
    "quarentena": f"{RAIZ_SQUAD}/grupo2/quarantine/ecommerce_itens_pedido",
    "produtos":   f"{RAIZ_SQUAD}/silver/ecommerce_produtos",
    "categorias": f"{RAIZ_SQUAD}/silver/ecommerce_categorias",
}

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def como_bigint(c):
    """Converte qualquer tipo (inclusive '1.0') para bigint; valor inválido vira nulo."""
    return F.expr(f"try_cast(try_cast(try_cast(`{c}` AS STRING) AS DOUBLE) AS BIGINT)")

## 1. Leitura
Da Silver de itens ficam só as colunas de negócio (as `flag_*`, receitas e colunas de partição são da Fase 1). O catálogo resolve a **categoria raiz** de cada SKU: a hierarquia tem 2 níveis (Categoria → Subcategoria).

In [ ]:
itens = ler_delta(FONTES["itens"]).select(
    "id_item_pedido", "id_pedido", F.trim("sku").alias("sku"),
    "quantidade", "preco_unitario", "desconto_aplicado", "dt_pedido")

cat = ler_delta(FONTES["categorias"]).select(
    como_bigint("id_categoria").alias("id_categoria"),
    F.trim("nome_categoria").alias("nome_categoria"),
    como_bigint("id_categoria_pai").alias("id_pai"))
nomes_raiz = cat.select(col("id_categoria").alias("id_raiz"), col("nome_categoria").alias("categoria_raiz"))
cat = (cat.withColumn("id_raiz", F.coalesce("id_pai", "id_categoria"))
          .join(nomes_raiz, "id_raiz", "left")
          .select("id_categoria", col("nome_categoria").alias("subcategoria"), "categoria_raiz"))

catalogo = (ler_delta(FONTES["produtos"])
            .select(F.trim("sku").alias("sku"), como_bigint("id_categoria").alias("id_categoria"))
            .dropDuplicates(["sku"])
            .join(cat, "id_categoria", "left"))

itens_cat = itens.join(F.broadcast(catalogo), "sku", "left")
print(f"Catálogo: {catalogo.count():,} SKUs | {catalogo.select('categoria_raiz').distinct().count()} categorias raiz "
      f"| {catalogo.select('subcategoria').distinct().count()} subcategorias")

## 2. Volume, período e qualidade
O Item2Vec precisa de muitas cestas. Esta seção conta cestas e SKUs, mostra o período e procura nulos, duplicatas e SKUs sem cadastro.

In [ ]:
colunas = ["id_pedido", "id_item_pedido", "sku", "quantidade", "preco_unitario", "desconto_aplicado", "dt_pedido"]
(itens_cat.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("id_pedido").alias("cestas"),
    F.countDistinct("sku").alias("skus"),
    F.min("dt_pedido").alias("de"), F.max("dt_pedido").alias("ate"),
    F.countDistinct(F.to_date("dt_pedido")).alias("dias_com_venda"),
    (F.count("*") - F.countDistinct("id_item_pedido")).alias("id_item_duplicado"),
    (F.count("*") - F.countDistinct("id_pedido", "sku")).alias("sku_repetido_no_pedido"),
    F.sum(col("categoria_raiz").isNull().cast("int")).alias("linhas_sem_catalogo"),
    *[F.sum(col(c).isNull().cast("int")).alias(f"nulos_{c}") for c in colunas])
 .show(truncate=False, vertical=True))

print("Cestas por dia:")
(itens.groupBy(F.to_date("dt_pedido").alias("dia")).agg(F.countDistinct("id_pedido").alias("cestas"))
      .orderBy("dia").show(40))

A Silver já filtra o que é tecnicamente inválido. O tamanho da **quarentena** mostra quanto ficou de fora.

In [ ]:
try:
    q = ler_delta(FONTES["quarentena"])
    print(f"Quarentena: {q.count():,} linhas")
    q.groupBy("quarantine_reason").count().orderBy(F.desc("count")).show(10, truncate=False)
except Exception as e:
    print(f"Quarentena não encontrada ({type(e).__name__})")

## 3. Distribuições e outliers das colunas numéricas
O Item2Vec **não usa** preço, quantidade nem desconto: ele só olha quais SKUs estão juntos na cesta. Mesmo assim, a distribuição dessas colunas mostra se há pedidos anômalos (ex.: quantidade gigante, típica de compra para revenda) que distorceriam as cestas. Outlier aqui segue a regra do IQR (acima de Q3 + 1,5 × IQR).

In [ ]:
def resumo_numerico(df, coluna):
    q1, med, q3, p99 = df.select(F.percentile_approx(coluna, [0.25, 0.5, 0.75, 0.99], 1000)).first()[0]
    limite = q3 + 1.5 * (q3 - q1)
    return (df.agg(lit(coluna).alias("coluna"), F.min(coluna).alias("min"), lit(q1).alias("q1"), lit(med).alias("mediana"),
                   lit(q3).alias("q3"), lit(p99).alias("p99"), F.max(coluna).alias("max"),
                   F.round(F.avg(coluna), 2).alias("media"),
                   F.round(100 * F.avg((col(coluna) > limite).cast("int")), 2).alias("pct_outlier_iqr")))

base_num = itens.withColumn("pct_desconto", F.round(100 * col("desconto_aplicado") / col("preco_unitario"), 2))
resumo = resumo_numerico(base_num, "quantidade")
for c in ["preco_unitario", "pct_desconto"]:
    resumo = resumo.unionByName(resumo_numerico(base_num, c))
resumo.show(truncate=False)

## 4. Tamanho das cestas
Uma cesta com **1 SKU** não ensina nada ao Item2Vec (não há "vizinho") e não serve para o teste (remover o único item deixa o carrinho vazio). O tamanho da cesta também define a `window` do Word2Vec no notebook 02.

In [ ]:
cestas = (itens_cat.groupBy("id_pedido").agg(
    F.min("dt_pedido").alias("dt_pedido"),
    F.countDistinct("sku").alias("n_skus"),
    F.sum("quantidade").alias("qtd_total"),
    F.round(F.sum(col("preco_unitario") * col("quantidade")), 2).alias("valor_bruto"),
    F.round(F.sum(col("desconto_aplicado") * col("quantidade")), 2).alias("desconto_total"),
    F.countDistinct("categoria_raiz").alias("n_categorias_raiz"))
  .withColumn("hora", F.hour("dt_pedido"))
  .withColumn("dia_semana", F.dayofweek("dt_pedido")))

(cestas.agg(
    F.count("*").alias("cestas"), F.round(F.avg("n_skus"), 1).alias("media_skus"),
    F.percentile_approx("n_skus", [0.05, 0.25, 0.5, 0.75, 0.95], 1000).alias("p05_p25_p50_p75_p95"),
    F.max("n_skus").alias("max_skus"),
    F.sum((col("n_skus") == 1).cast("int")).alias("cestas_1_sku"),
    F.round(100 * F.avg((col("n_skus") == 1).cast("int")), 1).alias("pct_1_sku"),
    F.round(F.avg(F.when(col("n_skus") >= 2, col("n_categorias_raiz"))), 2).alias("media_categorias_raiz_cesta_2mais"))
 .show(truncate=False))

dist = cestas.groupBy("n_skus").count().orderBy("n_skus").toPandas()
plt.figure(figsize=(9, 3.8))
plt.bar(dist.n_skus, dist["count"], color="#3a6ea5")
plt.title("SKUs distintos por cesta"); plt.xlabel("SKUs na cesta"); plt.ylabel("cestas")
plt.tight_layout(); plt.show()

## 5. Correlação entre as variáveis da cesta
Matriz de Pearson entre as variáveis numéricas de cada cesta. Ela mostra **como as cestas se comportam** (cesta maior → mais categorias? mais desconto? depende da hora?).

Para o Item2Vec, a "correlação" que importa é **entre produtos**: quais SKUs aparecem juntos. Ela é medida na seção 7 com o *lift*, que é a forma de correlação adequada para variáveis categóricas como o SKU.

In [ ]:
vars_cesta = ["n_skus", "qtd_total", "valor_bruto", "desconto_total", "n_categorias_raiz", "hora", "dia_semana"]
corr = cestas.select(*vars_cesta).toPandas().corr()   # 1 linha por cesta (~1,8 mil)

plt.figure(figsize=(7.5, 6))
plt.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.xticks(range(len(vars_cesta)), vars_cesta, rotation=45, ha="right"); plt.yticks(range(len(vars_cesta)), vars_cesta)
for i in range(len(vars_cesta)):
    for j in range(len(vars_cesta)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(); plt.title("Correlação (Pearson) entre variáveis da cesta"); plt.tight_layout(); plt.show()

## 6. Popularidade dos SKUs e tamanho do vocabulário
O Word2Vec descarta SKUs com menos de `min_count` ocorrências, e SKUs raros geram vetores ruins. Com poucos dados de stream, esse é o principal ponto de atenção: a tabela mostra, para cada `min_count`, quantos SKUs ficam no vocabulário e que fração das vendas eles cobrem.

In [ ]:
freq = itens.select("id_pedido", "sku").distinct().groupBy("sku").agg(F.count("*").alias("cestas_com_sku"))

freq.select(F.min("cestas_com_sku").alias("min"),
            F.expr("percentile_approx(cestas_com_sku, 0.5)").alias("mediana"),
            F.max("cestas_com_sku").alias("max"),
            F.sum((col("cestas_com_sku") == 1).cast("int")).alias("skus_vistos_1_vez")).show()

total_skus, total_ocorr = freq.agg(F.count("*"), F.sum("cestas_com_sku")).first()
(freq.crossJoin(spark.range(1, 11).select(col("id").alias("min_count")))
     .filter(col("cestas_com_sku") >= col("min_count"))
     .groupBy("min_count").agg(F.count("*").alias("skus_no_vocabulario"),
                               F.sum("cestas_com_sku").alias("ocorrencias"))
     .withColumn("pct_skus", F.round(100 * col("skus_no_vocabulario") / total_skus, 1))
     .withColumn("pct_vendas_cobertas", F.round(100 * col("ocorrencias") / total_ocorr, 1))
     .drop("ocorrencias").orderBy("min_count").show())

pdf = freq.orderBy(F.desc("cestas_com_sku")).toPandas()
plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, len(pdf) + 1), pdf.cestas_com_sku)
plt.yscale("log"); plt.xlabel("SKU (ordenado por popularidade)"); plt.ylabel("cestas com o SKU (log)")
plt.title("Cauda longa da popularidade dos SKUs"); plt.tight_layout(); plt.show()

## 7. Co-ocorrência: os produtos são comprados juntos por acaso?
Se as cestas fossem aleatórias, **nenhum** modelo de recomendação aprenderia algo. Com poucas cestas, o lift de cada par isolado não é confiável (um par visto 1 vez já parece "forte"), então o teste é feito **por categoria**:

* **observado** = fração dos pares dentro das cestas em que os dois itens são da mesma categoria;
* **acaso** = a mesma fração se os itens fossem sorteados = Σ (participação da categoria)²;
* **lift** = observado / acaso. Acima de 1, os clientes combinam produtos da mesma categoria mais do que o acaso explica, e é essa estrutura que os embeddings vão capturar.

In [ ]:
base_pares = itens_cat.select("id_pedido", "sku", "subcategoria", "categoria_raiz").distinct()
a, b = base_pares.alias("a"), base_pares.alias("b")
pares = a.join(b, (col("a.id_pedido") == col("b.id_pedido")) & (col("a.sku") < col("b.sku"))).select(
    col("a.sku").alias("sku_a"), col("b.sku").alias("sku_b"),
    col("a.subcategoria").alias("sub_a"), col("b.subcategoria").alias("sub_b"),
    col("a.categoria_raiz").alias("raiz_a"), col("b.categoria_raiz").alias("raiz_b"))

for nivel in ["categoria_raiz", "subcategoria"]:
    sufixo = "raiz" if nivel == "categoria_raiz" else "sub"
    observado = pares.agg(F.avg((col(f"{sufixo}_a") == col(f"{sufixo}_b")).cast("int"))).first()[0]
    part = base_pares.groupBy(nivel).count().withColumn("p", col("count") / F.sum("count").over(Window.partitionBy()))
    acaso = part.agg(F.sum(col("p") ** 2)).first()[0]
    print(f"{nivel:15} observado {observado:6.1%} | acaso {acaso:6.1%} | lift {observado / acaso:4.2f}")

pares_sup = pares.groupBy("sku_a", "sku_b").count()
print(f"\nPares distintos: {pares_sup.count():,} | vistos 2+ vezes: {pares_sup.filter('count >= 2').count():,} "
      f"| maior suporte: {pares_sup.agg(F.max('count')).first()[0]}")

### 7.1. Matriz de lift entre categorias raiz
É a "matriz de correlação" entre categorias: cada célula compara quantos pares (A, B) existem nas cestas com quantos existiriam ao acaso. A diagonal forte mostra afinidade dentro da categoria; fora da diagonal aparecem as combinações entre categorias (ex.: bebidas + petiscos).

In [ ]:
m = pares.groupBy("raiz_a", "raiz_b").count().toPandas()
part = base_pares.groupBy("categoria_raiz").count().toPandas().set_index("categoria_raiz")["count"]
p = part / part.sum()
raizes = list(p.sort_values(ascending=False).index)
idx = {r: i for i, r in enumerate(raizes)}
obs = np.zeros((len(raizes),) * 2)
for _, r in m.iterrows():                    # matriz simétrica: (A, B) e (B, A) contam no mesmo par
    i, j = idx[r.raiz_a], idx[r.raiz_b]
    obs[i, j] += r["count"]
    if i != j:
        obs[j, i] += r["count"]
total = obs[np.triu_indices(len(raizes))].sum()
pv = p[raizes].values
esp = total * np.outer(pv, pv) * np.where(np.eye(len(raizes)) == 1, 1, 2)   # fora da diagonal o par pode sair em 2 ordens
lift_cat = obs / esp

plt.figure(figsize=(9, 7.5))
plt.imshow(lift_cat, cmap="YlOrRd")
plt.xticks(range(len(raizes)), raizes, rotation=60, ha="right"); plt.yticks(range(len(raizes)), raizes)
for i in range(len(raizes)):
    for j in range(len(raizes)):
        plt.text(j, i, f"{lift_cat[i, j]:.1f}", ha="center", va="center", fontsize=7)
plt.colorbar(label="lift"); plt.title("Lift entre categorias raiz (stream)"); plt.tight_layout(); plt.show()

## 8. Viabilidade do hold-out
Protocolo para comparar com o Modelo A (FP-Growth, Lucas): **teste = últimos ~20% das cestas por data**, só cestas com 2+ SKUs; remove-se 1 item e mede-se se o modelo o recupera no top-5. O treino usa tudo o que veio antes do corte.

O **teto do Hit@5** é a fração de itens do teste cujo SKU apareceu no treino: SKU nunca visto não pode ser recomendado pelo Item2Vec. A mesma conta por subcategoria mostra quanto o **fallback por categoria** consegue cobrir.

In [ ]:
validas = cestas.filter(col("n_skus") >= 2)
corte = validas.agg(F.expr("percentile_approx(dt_pedido, 0.8, 10000)")).first()[0]
ids_teste = validas.filter(col("dt_pedido") > corte).select("id_pedido")
treino = itens_cat.join(ids_teste, "id_pedido", "left_anti").filter(col("dt_pedido") <= corte)
teste = itens_cat.join(ids_teste, "id_pedido")

vocab_sku = treino.select("sku").distinct().withColumn("sku_no_treino", lit(True))
vocab_sub = treino.select("subcategoria").distinct().withColumn("sub_no_treino", lit(True))
print(f"Data de corte: {corte}")
(teste.join(F.broadcast(vocab_sku), "sku", "left").join(F.broadcast(vocab_sub), "subcategoria", "left")
      .fillna(False, ["sku_no_treino", "sub_no_treino"])
      .agg(F.countDistinct("id_pedido").alias("cestas_teste"), F.count("*").alias("itens_teste"),
           F.round(100 * F.avg(col("sku_no_treino").cast("int")), 1).alias("teto_hit_sku_pct"),
           F.round(100 * F.avg(col("sub_no_treino").cast("int")), 1).alias("cobertura_fallback_subcategoria_pct"))
      .show(truncate=False))
print(f"Cestas de treino (2+ SKUs): {validas.filter(col('dt_pedido') <= corte).count():,}")

## 9. Veredito

> Números medidos direto na Silver em 12/11/2026, antes desta execução. Conferir com as saídas acima.

| Pergunta | Resposta |
|---|---|
| **Volume** | 9.294 linhas, **1.834 cestas**, 2.512 SKUs, de 21/09 a 15/10/2026 (~3,5 semanas). |
| **Qualidade** | Sem nulos e sem SKU fora do catálogo: a Silver da Fase 1 já fez a limpeza e mandou o resto para a quarentena. |
| **Cestas** | Média de 5,1 SKUs (mediana 4, p95 13, máx. 24). **152 cestas com 1 SKU** (8%) ficam fora do treino e do teste: sobram **1.682** cestas úteis. Cada cesta mistura em média 2 categorias raiz. |
| **Cauda longa** | Mediana de 3 cestas por SKU; **763 SKUs (30%) aparecem só 1 vez**. Com `min_count = 2`, o vocabulário fica com 1.749 SKUs e cobre 92% das vendas; com `min_count = 5`, cai para 657 SKUs e 58%. |
| **Sinal** | Dois itens da mesma cesta são da **mesma categoria raiz em 42,6%** dos pares, contra 15,3% ao acaso (**lift 2,78**); mesma subcategoria: 6,2% contra 2,2% (**lift 2,83**). Há estrutura de compra para o modelo aprender. |
| **Hold-out** | Corte em 08/10/2026: **336 cestas** e 1.582 itens de teste. **88,6%** dos itens de teste têm SKU visto no treino (teto do Hit@5); por subcategoria, a cobertura é de 99,9%. |

**Conclusão: dá para treinar o Item2Vec só com o stream, mas a base é pequena e esparsa.** O sinal existe, mas é mais forte no nível de categoria do que de SKU, porque a maioria dos pares de SKUs aparece uma única vez. Decisões levadas para o notebook 02 (features):
1. **Descartar cestas com 1 SKU** e contar cada SKU uma vez por pedido (a cesta é um conjunto).
2. **`min_count` baixo (2)** para manter 92% das vendas no vocabulário; SKUs fora dele caem no **fallback por subcategoria** (mais vendidos da subcategoria do carrinho).
3. **`window` do tamanho da cesta** (até 24) e **embedding pequeno (16–32 dimensões)**: com 1,7 mil cestas, vetores grandes decorariam o treino em vez de generalizar.
4. Preço, quantidade e desconto **não entram** no Item2Vec (ele só usa a sequência de SKUs); servem para a EDA e para desempatar o fallback por popularidade.
5. **Baseline obrigatório:** top-5 mais vendidos. O Item2Vec só se justifica se superar esse baseline no mesmo hold-out.